# Showcase

A CodeLens-style execution visualizer that uses **Philip Guo's real Online Python Tutor
frontend** (`pytutor.js`). The execution trace is generated in the live kernel by Guo's own
`pg_logger` (vendored, MIT), so it is in his **exact schema**, and rendered by `pytutor.js`
inside a self-contained `<iframe>` — step through with the forward/back buttons, inspect the
call stack, and watch the heap drawn as reference diagrams with pointer arrows.

Everything is vendored under `codelens_widget/vendor/`, so it runs **fully offline** and the
same way in VS Code, JupyterLab, Notebook 7, and Colab.

> **Setup:** keep the `codelens_widget/` package folder next to this notebook, and
> `pip install anywidget`. Online Python Tutor is © Philip J. Guo, MIT-licensed.

In [1]:
from codelens_widget import CodeLens

## 1. Insertion sort

The canonical CodeLens example. Step through with the buttons; the in-place swaps mutate the
same list object on the heap.

In [2]:
%%codelens
def insertion_sort(a):
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a

data = [5, 2, 9, 1]
insertion_sort(data)

In [ ]:
CodeLens("""
def insertion_sort(a):
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a

data = [5, 2, 9, 1]
insertion_sort(data)
""")

## 2. Aliasing — the whole point of reference diagrams

`y = x` does **not** copy the list; both names point at the same object, so `y.append(4)` is
visible through `x`. Python Tutor draws both variables with arrows to one heap box.

In [ ]:
CodeLens("""
x = [1, 2, 3]
y = x
y.append(4)
print(x)
""")

## 3. Objects & linked structure

Instances render as boxes of fields; references between them become arrows — here a tiny
linked list.

In [ ]:
CodeLens("""
class Node:
    def __init__(self, value, nxt=None):
        self.value = value
        self.next = nxt

head = Node(1, Node(2, Node(3)))
cur = head
total = 0
while cur is not None:
    total += cur.value
    cur = cur.next
""", height=560)

## 4. Recursion — the call stack grows

Each recursive call pushes a frame onto the stack; watch them stack up and unwind.

In [ ]:
CodeLens("""
def fact(n):
    if n <= 1:
        return 1
    return n * fact(n - 1)

fact(4)
""")

## 5. Nested containers & dicts

In [ ]:
CodeLens("""
matrix = [[1, 2], [3, 4]]
lookup = {"a": matrix[0], "b": matrix[1]}
matrix[0][1] = 99
""")

## 6. The `%%codelens` cell magic

Importing the package registers a cell magic, so you can visualize a whole cell directly —
the closest analog to Runestone's `.. codelens::` directive.

In [ ]:
%%codelens
nums = [3, 1, 2]
nums.sort()
squared = [n * n for n in nums]
print(squared)

## 7. Exceptions are traced too

An uncaught error becomes a final step in the trace.

In [ ]:
CodeLens("""
def risky(x):
    return 10 / x

risky(0)
""")

## 8. Self-test — Guo's exact schema (headless)

Rendering needs a browser, but the **trace data** can be validated headlessly. This asserts the
trace matches Online Python Tutor's schema: the per-step keys, `REF`-based aliasing, heap object
tags (`LIST`, `INSTANCE`, …), the call stack, and JSON-serializability (the contract with
`pytutor.js`).

In [ ]:
import json
from codelens_widget import trace_code

REQUIRED = {"line", "event", "func_name", "globals", "ordered_globals",
            "stack_to_render", "heap", "stdout"}

t = trace_code("""
x = [1, 2, 3]
y = x
y.append(4)
""")
assert set(t.keys()) == {"code", "trace"}
assert all(REQUIRED <= set(p.keys()) for p in t["trace"])
assert {"step_line"} <= {p["event"] for p in t["trace"]}

last = t["trace"][-1]
gx, gy = last["globals"]["x"], last["globals"]["y"]
assert gx[0] == "REF" and gx == gy, "aliasing: x and y must share one heap id"
assert last["heap"][gx[1]][0] == "LIST"
assert last["heap"][gx[1]][1:] == [1, 2, 3, 4]

# instance object encodes as INSTANCE with a class name + fields
ti = trace_code("class P:\n    def __init__(s): s.x = 1\np = P()")
li = ti["trace"][-1]
inst = li["heap"][li["globals"]["p"][1]]
assert inst[0] == "INSTANCE" and inst[1] == "P"

# recursion produces a growing stack_to_render
tr = trace_code("def f(n):\n    if n<=1: return 1\n    return n*f(n-1)\nf(4)")
assert max(len(p["stack_to_render"]) for p in tr["trace"]) >= 4

# whole thing is JSON-serializable (heap int keys -> string keys, as pytutor.js expects)
rt = json.loads(json.dumps(t))
assert all(isinstance(k, str) for k in rt["trace"][-1]["heap"])

print("All schema self-tests passed.")
print("insertion-trace points:", len(t["trace"]),
      "| events:", sorted({p["event"] for p in t["trace"]}))